# Circuit 4 — Grover's Algorithm (3 qubits, target |111⟩)

**What it does:** Searches an 8-element space (3 qubits) for the marked state |111⟩
using amplitude amplification. Two Grover iterations are applied — the theoretically
optimal count for N=8 (`floor(π/4 · √8) = 2`), giving a success probability near 1.

Each iteration consists of:
- **Oracle** — a single CCZ gate that flips the phase of |111⟩
- **Diffuser** — H⊗3 · X⊗3 · CCZ · X⊗3 · H⊗3 (reflects about the uniform superposition)

**Statevector path test:** CCZ (Toffoli) is non-Clifford, so this notebook intentionally
skips the Stim/ManyShotRunner path and uses `TrajectoryBackend.run_aggregate()` for
everything. This proves that the statevector simulator feeds directly into the
heatmap, interactive plot, and GIF visualization pipeline — no Clifford stand-in needed.

`run_aggregate()` returns an `AggregateResult` (same dataclass as `ManyShotRunner`)
with `final_state` already populated, so one call covers both the heatmap pass and
the purity computation.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import noisiq as nq
from noisiq.backends import TrajectoryBackend
from noisiq.noise import fill_idle_with_identities
from noisiq.visualization import (
    Visualizer,
    export_gif,
    interactive_heatmap,
    plot_error_heatmap,
    per_qubit_purities,
    add_purity_panel,
)

os.makedirs("outputs", exist_ok=True)

profile    = nq.noise.get_hardware("ibm_eagle_r3")
gate_times = profile.gate_times

# TrajectoryBackend is statevector-based (slower per shot than Stim).
# 3 qubits → 8×8 density matrix, so 2000 shots is still fast.
N_SHOTS = 2000

print(f"noisiq {nq.__version__}")
print(profile.describe())

In [ ]:
def print_purities(rho, n_qubits, label=""):
    """Compute and print per-qubit purities from a density matrix."""
    purities = per_qubit_purities(rho, n_qubits)
    print(f"\n{'─'*50}")
    print(f"Per-qubit purity  [{label}]")
    for q, p in enumerate(purities):
        print(f"  q{q:>2d}  Tr(ρ²) = {p:.4f}  {'█' * int(p * 20)}")
    print(f"{'─'*50}\n")
    return purities


def visualize_circuit(circuit, result, noise_pauli, rho, purities,
                      label, gif_name):
    """Static heatmap + interactive heatmap + GIF export."""
    fig = plot_error_heatmap(
        result, circuit, noise_config=noise_pauli,
        title=f"IBM Eagle r3 · {label}",
    )
    add_purity_panel(fig, fig.axes[0], rho, circuit.n_qubits)
    plt.show()

    interactive_heatmap(
        result, circuit, noise_config=noise_pauli,
        display_mode="annotate",
        title=f"IBM Eagle r3 · {label} [interactive]",
    )
    plt.show()

    viz = Visualizer(circuit)
    viz.many_shot_result = result
    gif_path = f"outputs/{gif_name}.gif"
    export_gif(viz, gif_path, purities=purities)
    print(f"GIF → {gif_path}")


print("Helpers ready.")

In [ ]:
N_GRV               = 3
N_GROVER_ITERATIONS = 2  # optimal for N=8 (3 qubits): floor(π/4 · √8) = 2


def build_grover_circuit(n: int, n_iter: int) -> nq.Circuit:
    """
    Grover circuit targeting |111⟩.
    Oracle: CCZ(0,1,2) flips the phase of |111⟩.
    Diffuser: H⊗n · X⊗n · CCZ · X⊗n · H⊗n reflects about the uniform superposition.
    """
    c = nq.Circuit(n_qubits=n, name=f"grover_{n}q")

    # Uniform superposition
    for q in range(n):
        c.h(q)

    for _ in range(n_iter):
        # Oracle: phase flip on |111⟩
        c.ccz(0, 1, 2)

        # Diffuser: reflect about the uniform superposition
        for q in range(n):
            c.h(q)
        for q in range(n):
            c.x(q)
        c.ccz(0, 1, 2)
        for q in range(n):
            c.x(q)
        for q in range(n):
            c.h(q)

    return c


circuit_grv = fill_idle_with_identities(
    build_grover_circuit(N_GRV, N_GROVER_ITERATIONS), gate_times
)
print(f"Grover circuit ops (idle-filled): {len(circuit_grv.operations)}")

In [ ]:
noise_grv = profile.to_noise_model(
    circuit_grv, mode="t2", representation="pauli_twirl",
)

# run_aggregate uses the statevector simulator (TrajectoryBackend) rather than Stim.
# It returns an AggregateResult — the same dataclass ManyShotRunner produces —
# so all downstream visualization calls are identical.
# final_state (averaged density matrix) is populated on the same call,
# so no separate purity pass is needed.
result_grv = TrajectoryBackend().run_aggregate(
    circuit_grv, noise_model=noise_grv, n_shots=N_SHOTS, seed=42,
)
print(f"run_aggregate done  zero-error fraction: {result_grv.zero_error_fraction:.4f}")

rho_grv = result_grv.final_state
pur_grv = print_purities(rho_grv, N_GRV, "Grover-3")

In [ ]:
visualize_circuit(
    circuit_grv, result_grv, noise_grv, rho_grv, pur_grv,
    label=f"Grover (3q, target |111⟩, {N_GROVER_ITERATIONS} iterations)",
    gif_name="grover_3q",
)